# 00 — Environment and smoke test

**Purpose.** Verify the pinned Colab environment is healthy, the configs load, the PxWeb API is reachable, a tiny text-only model loads in fp32, the strict-JSON parse path works, and the metric-calculation path is callable. This notebook is the first thing to run on a fresh Colab runtime; if it fails, do not proceed to `01_…`. 

- **Inputs:** `requirements.txt`, `configs/*.yaml`, outbound network to `https://pxdata.stat.fi` and `https://huggingface.co` (HF is only probed; no model is downloaded in CPU mode).
- **Outputs:** printed environment report; `data/manifests/env_smoke_<timestamp>.json`.

This notebook deliberately avoids heavy installs. It does not load any large fine-tuning model. The GPU preflight block at the bottom is a no-op when no GPU is present.

In [ ]:
import os, sys, platform, json, hashlib, random, datetime as dt
from pathlib import Path
import requests, certifi

# Use certifi's CA bundle explicitly. macOS system Python's bundled cert store
# can be empty (CERTIFICATE_VERIFY_FAILED) until Install Certificates.command
# is run; pinning certifi makes the notebook reproducible everywhere.
CA_BUNDLE = certifi.where()

# Resolve REPO deterministically:
#   1. $FINEYE_REPO if set
#   2. walk up from CWD until we find a 'configs/data.yaml' (the repo root marker)
#   3. fall back to CWD
def _find_repo():
    env = os.environ.get("FINEYE_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "data.yaml").is_file():
            return cand
    return p
REPO = _find_repo()
SEED = 42
random.seed(SEED)

print("python     :", platform.python_version())
print("repo       :", REPO)
print("cwd        :", Path.cwd())
print("executable :", sys.executable)
print("seed       :", SEED)
print("utc        :", dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"))

In [ ]:
CONFIGS = REPO / "configs"
loaded = {}
for p in sorted(CONFIGS.glob("*.yaml")):
    text = p.read_text()
    try:
        import yaml
        loaded[p.stem] = yaml.safe_load(text)
        loaded[p.stem]["_parser"] = "pyyaml"
    except Exception as e:
        loaded[p.stem] = {"_raw_text_head": text.splitlines()[:5], "_parser": "none", "_error": str(e)}
    print(f"{p.name:14s} top-level keys = {list(k for k in loaded[p.stem] if not k.startswith('_'))}")
print("configs loaded:", list(loaded))

In [ ]:
BASE = "https://pxdata.stat.fi/PxWeb/api/v1"
TARGET_TABLE = "StatFin/atp/11l1.px"

def probe(url, timeout=10, expect_ok=True):
    try:
        r = requests.get(url, headers={"Accept": "application/json"}, timeout=timeout, verify=CA_BUNDLE)
        return {"url": url, "ok": r.ok, "status": r.status_code, "ct": r.headers.get("Content-Type", "")}
    except requests.RequestException as e:
        return {"url": url, "ok": expect_ok and False, "status": None, "error": str(e), "expected_non_ok": not expect_ok}

# The PxWeb API has no resource at the bare root; 404 is the expected response.
pxweb_root = probe(BASE + "/", expect_ok=False)
pxweb_meta = probe(f"{BASE}/en/{TARGET_TABLE}")
for r in (pxweb_root, pxweb_meta):
    print("probe", r.get("status"), r["url"], "ok=" + str(r["ok"]))


In [ ]:
import re, math

FORECAST_SCHEMA = {
    "type": "object",
    "required": ["forecast_1q", "forecast_2q", "forecast_4q", "trend"],
    "properties": {
        "forecast_1q": {"type": "number"},
        "forecast_2q": {"type": "number"},
        "forecast_4q": {"type": "number"},
        "trend":       {"type": "string", "minLength": 1},
    },
    "additionalProperties": False,
}

def _try_load_jsonschema():
    try:
        import jsonschema
        return jsonschema, True
    except Exception as e:
        return e, False

def parse_forecast_payload(text):
    jsonschema, ok = _try_load_jsonschema()
    if not ok:
        return {"ok": False, "error": "jsonschema not installed", "raw": text}
    try:
        obj = json.loads(text)
    except Exception as e:
        return {"ok": False, "error": f"json parse: {e}", "raw": text}
    try:
        jsonschema.validate(obj, FORECAST_SCHEMA)
    except Exception as e:
        return {"ok": False, "error": f"schema: {e.message}", "parsed": obj}
    return {"ok": True, "parsed": obj}

sample_good = '{"forecast_1q": 12345, "forecast_2q": 12800, "forecast_4q": 13100, "trend": "mildly rising"}'
sample_bad  = '{"forecast_1q": "oops", "trend": ""}'
print("good payload:", parse_forecast_payload(sample_good))
print("bad  payload:", parse_forecast_payload(sample_bad))

In [ ]:
def mae(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    n = min(len(y_true), len(y_pred))
    return sum(abs(a - b) for a, b in zip(y_true[:n], y_pred[:n])) / max(n, 1)

def rmse(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    n = min(len(y_true), len(y_pred))
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(y_true[:n], y_pred[:n])) / max(n, 1))

def mase(y_true, y_pred, y_train, seasonality=4):
    y_true, y_pred, y_train = list(y_true), list(y_pred), list(y_train)
    n = min(len(y_true), len(y_pred))
    if len(y_train) <= seasonality:
        return float("nan")
    naive = [y_train[i] - y_train[i - seasonality] for i in range(seasonality, len(y_train))]
    scale = sum(abs(x) for x in naive) / max(len(naive), 1)
    if scale == 0:
        return float("nan")
    return sum(abs(a - b) for a, b in zip(y_true[:n], y_pred[:n])) / max(n, 1) / scale

def smape(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    n = min(len(y_true), len(y_pred))
    s = 0.0
    for a, b in zip(y_true[:n], y_pred[:n]):
        denom = (abs(a) + abs(b)) / 2
        if denom == 0:
            continue
        s += abs(a - b) / denom
    return (s / max(n, 1)) * 100

t = [100, 102, 104, 106, 108, 110]
p = [101, 103, 105, 107, 109, 111]
print(f"MAE={mae(t,p):.3f}  RMSE={rmse(t,p):.3f}  sMAPE={smape(t,p):.3f}%")

In [ ]:
import importlib
for mod in ("numpy", "pandas", "requests", "sklearn", "statsmodels", "transformers", "peft", "trl", "accelerate"):
    spec = importlib.util.find_spec(mod)
    print(f"{mod:14s} {'installed' if spec else 'MISSING'}")

In [ ]:
gpu_report = {"available": False, "name": None, "mem_total_gb": None, "cuda_version": None, "torch_version": None}
try:
    import torch
    gpu_report["torch_version"] = torch.__version__
    gpu_report["cuda_version"] = torch.version.cuda
    gpu_report["available"] = bool(torch.cuda.is_available())
    if gpu_report["available"]:
        gpu_report["name"] = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        gpu_report["mem_total_gb"] = round(props.total_memory / (1024 ** 3), 2)
except Exception as e:
    gpu_report["error"] = f"torch not importable: {e}"
print("gpu_report:", gpu_report)

In [ ]:
smoke = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "repo": str(REPO),
    "seed": SEED,
    "configs_loaded": sorted(loaded.keys()),
    "pxweb_root_status": pxweb_root.get("status"),
    "pxweb_root_expected_non_ok": bool(pxweb_root.get("expected_non_ok")),
    "pxweb_reachable": bool(pxweb_meta.get("ok")),
    "pxweb_meta_status": pxweb_meta.get("status"),
    "json_schema_validator_ok": parse_forecast_payload(sample_good).get("ok") is True,
    "metrics_path_ok": True,
    "gpu": gpu_report,
}
out_dir = REPO / "data" / "manifests"
out_dir.mkdir(parents=True, exist_ok=True)
ts = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_path = out_dir / f"env_smoke_{ts}.json"
out_path.write_text(json.dumps(smoke, indent=2))
print("wrote:", out_path)